# NCAA Seed Prediction using TensorFlow Decision Forests

Adapted from the [House Prices TFDF notebook](https://www.kaggle.com/code/gusthema/house-prices-prediction-using-tfdf) that scored **0.11**.

Same approach: load data → feature engineering → `tfdf.keras.RandomForestModel` + `GradientBoostedTreesModel` → ensemble → predict.

**Key additions for NCAA:**
- Two-stage: non-tournament = 0, tournament-only TFDF regression
- TFDF hyperparameter tuner (automated search)
- Multi-model ensemble: TFDF RF + GBT + XGBoost + CatBoost
- Hungarian algorithm for optimal integer seed assignment

In [ ]:
# Install & Import (same as house-prices notebook)
!pip install -q tensorflow_decision_forests xgboost lightgbm catboost optuna

import tensorflow as tf
import tensorflow_decision_forests as tfdf
import pandas as pd
import numpy as np
import re, warnings
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.optimize import linear_sum_assignment, minimize
warnings.filterwarnings('ignore')
%matplotlib inline

print('TensorFlow v' + tf.__version__)
print('TensorFlow Decision Forests v' + tfdf.__version__)
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# Load data from Google Drive
from google.colab import drive, files
drive.mount('/content/drive')
!cp '/content/drive/MyDrive/'*.csv .
print('Data loaded from Google Drive')

In [ ]:
# Load the dataset (like the house-prices notebook)
dataset_df = pd.read_csv('NCAA_Seed_Training_Set2.0.csv')
test_data = pd.read_csv('NCAA_Seed_Test_Set2.0.csv')
sub_template = pd.read_csv('submission_template2.0.csv')

print('Full train dataset shape is {}'.format(dataset_df.shape))
print('Full test dataset shape is {}'.format(test_data.shape))
dataset_df.head(3)

In [ ]:
# Feature Engineering
# Parse W-L records and quadrant records into numeric features

def parse_wl(s):
    if pd.isna(s): return (np.nan, np.nan)
    m = re.search(r'(\d+)[^\d]+(\d+)', str(s))
    return (int(m.group(1)), int(m.group(2))) if m else (np.nan, np.nan)

def parse_q(s, idx=0):
    if pd.isna(s) or str(s).strip() == '': return np.nan
    m = re.search(r'(\d+)[^\d]+(\d+)', str(s))
    return int(m.group(idx+1)) if m else np.nan

# Parse numeric columns
for col in ['NET Rank','PrevNET','AvgOppNETRank','AvgOppNET','NETSOS','NETNonConfSOS']:
    for df in (dataset_df, test_data):
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')
dataset_df['Overall Seed'] = pd.to_numeric(dataset_df['Overall Seed'], errors='coerce')

# Parse W-L records
for col in ['WL','Conf.Record','Non-ConferenceRecord','RoadWL']:
    for df in (dataset_df, test_data):
        if col in df.columns:
            wl = df[col].apply(parse_wl)
            df[col+'_W'] = wl.apply(lambda x: x[0])
            df[col+'_L'] = wl.apply(lambda x: x[1])

# Parse quadrant records
for q in ['Quadrant1','Quadrant2','Quadrant3','Quadrant4']:
    for df in (dataset_df, test_data):
        df[q+'_W'] = df.get(q, pd.Series()).apply(lambda s: parse_q(s, 0))
        df[q+'_L'] = df.get(q, pd.Series()).apply(lambda s: parse_q(s, 1))

# Derived features
for df in (dataset_df, test_data):
    df['total_W'] = df['WL_W'].fillna(0); df['total_L'] = df['WL_L'].fillna(0)
    df['win_pct'] = df['total_W']/(df['total_W']+df['total_L']+1e-9)
    df['q1w'] = df['Quadrant1_W'].fillna(0); df['q1l'] = df['Quadrant1_L'].fillna(0)
    df['q2w'] = df['Quadrant2_W'].fillna(0); df['q2l'] = df['Quadrant2_L'].fillna(0)
    df['q3l'] = df['Quadrant3_L'].fillna(0); df['q4l'] = df['Quadrant4_L'].fillna(0)
    df['quality_wins'] = df['q1w']*2 + df['q2w']
    df['bad_losses'] = df['q3l'] + df['q4l']*2
    df['q1_pct'] = df['q1w']/(df['q1w']+df['q1l']+1e-9)
    df['conf_W'] = df['Conf.Record_W'].fillna(0); df['conf_L'] = df['Conf.Record_L'].fillna(0)
    df['conf_pct'] = df['conf_W']/(df['conf_W']+df['conf_L']+1e-9)
    df['road_W'] = df['RoadWL_W'].fillna(0); df['road_L'] = df['RoadWL_L'].fillna(0)
    df['road_pct'] = df['road_W']/(df['road_W']+df['road_L']+1e-9)
    df['nonconf_W'] = df['Non-ConferenceRecord_W'].fillna(0)
    df['nonconf_L'] = df['Non-ConferenceRecord_L'].fillna(0)
    df['nonconf_pct'] = df['nonconf_W']/(df['nonconf_W']+df['nonconf_L']+1e-9)
    df['NET'] = df['NET Rank'].fillna(200)
    df['PrevNET_val'] = df['PrevNET'].fillna(200)
    df['opp_NET'] = pd.to_numeric(
        df.get('AvgOppNETRank', df.get('AvgOppNET Rank', pd.Series(dtype=float))),
        errors='coerce').fillna(150)
    df['SOS'] = df['NETSOS'].fillna(150)
    df['SOS_NC'] = df['NETNonConfSOS'].fillna(150)
    df['is_AL'] = (df['Bid Type']=='AL').astype(float)
    df['is_AQ'] = (df['Bid Type']=='AQ').astype(float)
    df['winsm_losses'] = df['quality_wins'] - df['bad_losses']
    df['NET_delta'] = df['NET'] - df['PrevNET_val']
    df['q1_ratio'] = df['q1w'] / (df['q1w'] + df['q1l'] + df['q2w'] + df['q2l'] + 1e-9)

# Two-stage: separate tournament from non-tournament
train_tourn = dataset_df[dataset_df['Overall Seed'].notna()].copy()
test_tourn = test_data[test_data['Bid Type'].notna()].copy()

print(f'Tournament training teams: {len(train_tourn)}')
print(f'Tournament test teams: {len(test_tourn)}')
print(f'Non-tournament test: {len(test_data) - len(test_tourn)}')

In [ ]:
# Within-Season Features (critical: ~0.94 correlation with seed)

for season in dataset_df['Season'].unique():
    tr_m = train_tourn['Season']==season
    te_m = test_tourn['Season']==season
    all_nets = np.sort(np.concatenate([
        train_tourn.loc[tr_m,'NET'].values,
        test_tourn.loc[te_m,'NET'].values
    ]))
    for idx in train_tourn[tr_m].index:
        train_tourn.loc[idx,'tourn_net_rank'] = np.searchsorted(all_nets, train_tourn.loc[idx,'NET'])+1
    for idx in test_tourn[te_m].index:
        test_tourn.loc[idx,'tourn_net_rank'] = np.searchsorted(all_nets, test_tourn.loc[idx,'NET'])+1

    # Within-season relative features
    tr_teams = train_tourn[tr_m]
    for stat_col, new_col in [('NET','net_vs_avg'),('quality_wins','qw_vs_avg'),
                               ('bad_losses','bl_vs_avg'),('win_pct','wp_vs_avg')]:
        avg = tr_teams[stat_col].mean()
        for idx in train_tourn[tr_m].index:
            train_tourn.loc[idx,new_col] = train_tourn.loc[idx,stat_col] - avg
        for idx in test_tourn[te_m].index:
            test_tourn.loc[idx,new_col] = test_tourn.loc[idx,stat_col] - avg

    for conf in tr_teams['Conference'].unique():
        ct = tr_teams[tr_teams['Conference']==conf]
        avg_s = ct['Overall Seed'].mean() if len(ct)>0 else 34.5
        cnt = len(ct)
        for idx in train_tourn[(tr_m)&(train_tourn['Conference']==conf)].index:
            train_tourn.loc[idx,'conf_season_avg'] = avg_s
            train_tourn.loc[idx,'conf_season_cnt'] = cnt
        for idx in test_tourn[(te_m)&(test_tourn['Conference']==conf)].index:
            test_tourn.loc[idx,'conf_season_avg'] = avg_s
            test_tourn.loc[idx,'conf_season_cnt'] = cnt

# Conference target encoding
global_mean = train_tourn['Overall Seed'].mean()
cs = train_tourn.groupby('Conference')['Overall Seed'].agg(['mean','count'])
cs['enc'] = (cs['mean']*cs['count']+global_mean*5)/(cs['count']+5)
conf_enc_map = cs['enc'].to_dict()
for df in (train_tourn, test_tourn):
    df.loc[:,'conf_enc'] = df['Conference'].map(conf_enc_map).fillna(global_mean)

# Power conference flag
power_confs = ['Big Ten', 'SEC', 'Big 12', 'ACC', 'Big East']
for df in (train_tourn, test_tourn):
    df.loc[:,'is_power'] = df['Conference'].isin(power_confs).astype(float)

# Feature list
features = ['tourn_net_rank','NET','PrevNET_val','opp_NET','SOS','SOS_NC',
            'total_W','total_L','win_pct','q1w','q1l','q2w','q2l',
            'quality_wins','bad_losses','q1_pct','conf_W','conf_L','conf_pct',
            'road_W','road_L','road_pct','nonconf_pct',
            'is_AL','is_AQ','winsm_losses','NET_delta','q1_ratio',
            'net_vs_avg','qw_vs_avg','bl_vs_avg','wp_vs_avg',
            'conf_season_avg','conf_season_cnt','conf_enc','is_power']

LABEL = 'OverallSeed'  # No spaces (TFDF fix_feature_names issue)
train_tourn[LABEL] = train_tourn['Overall Seed']

for col in features:
    med = train_tourn[col].median()
    if pd.isna(med): med = 0
    train_tourn[col] = train_tourn[col].fillna(med)
    test_tourn[col] = test_tourn[col].fillna(med)

corr = np.corrcoef(train_tourn['tourn_net_rank'].values,
                    train_tourn[LABEL].values)[0,1]
print(f'Features: {len(features)}')
print(f'tourn_net_rank <-> seed correlation: {corr:.4f}')

In [ ]:
# Seed Distribution (like the house-prices SalePrice distribution plot)
print(train_tourn[LABEL].describe())
plt.figure(figsize=(9, 8))
sns.histplot(train_tourn[LABEL], color='g', bins=68, kde=True)
plt.xlabel('Overall Seed')
plt.title('Tournament Seed Distribution')
plt.show()

In [ ]:
# Prepare dataset for TFDF (same pattern as house-prices notebook)
train_df = train_tourn[features + [LABEL]].copy()
test_df = test_tourn[features].copy()

print('{} examples in training.'.format(len(train_df)))
train_df.head(3)

In [ ]:
# Convert to TF Datasets (same as house-prices notebook)
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_df, label=LABEL, task=tfdf.keras.Task.REGRESSION)
test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    test_df, task=tfdf.keras.Task.REGRESSION)

In [ ]:
# Create & Train Random Forest (same as house-prices notebook but with tuner)
# The house-prices notebook used defaults; here we use TFDF's built-in tuner

# Faster RF: no tuner, fewer trees
rf = tfdf.keras.RandomForestModel(
    task=tfdf.keras.Task.REGRESSION,
    num_trees=200, max_depth=16, min_examples=2,
    random_seed=42
)
rf.compile(metrics=['mse'])
rf.fit(x=train_ds)
print('Random Forest trained!')

In [ ]:
# Evaluate on OOB data (same as house-prices notebook)
inspector_rf = rf.make_inspector()
print('RF OOB evaluation:')
print(inspector_rf.evaluation())

# Plot RMSE vs number of trees
logs = inspector_rf.training_logs()
if logs:
    plt.figure(figsize=(9,5))
    plt.plot([log.num_trees for log in logs], [log.evaluation.rmse for log in logs])
    plt.xlabel('Number of trees')
    plt.ylabel('RMSE (out-of-bag)')
    plt.title('RF: RMSE vs Number of Trees')
    plt.show()

In [ ]:
# Faster GBT: no tuner, fewer trees
gbt = tfdf.keras.GradientBoostedTreesModel(
    task=tfdf.keras.Task.REGRESSION,
    num_trees=200, max_depth=6, shrinkage=0.05,
    subsample=0.8, l2_regularization=1.0,
    random_seed=42
)
gbt.compile(metrics=['mse'])
gbt.fit(x=train_ds)

inspector_gbt = gbt.make_inspector()
print('GBT evaluation:')
print(inspector_gbt.evaluation())

In [ ]:
# Train additional TFDF model variants with different seeds for diversity
rf2 = tfdf.keras.RandomForestModel(
    task=tfdf.keras.Task.REGRESSION,
    num_trees=1500, max_depth=16, min_examples=2,
    random_seed=123
)
rf2.compile(metrics=['mse'])
rf2.fit(x=train_ds)

gbt2 = tfdf.keras.GradientBoostedTreesModel(
    task=tfdf.keras.Task.REGRESSION,
    num_trees=800, max_depth=4, shrinkage=0.03,
    subsample=0.7, l2_regularization=1.0,
    random_seed=456
)
gbt2.compile(metrics=['mse'])
gbt2.fit(x=train_ds)

print('RF2 OOB:', rf2.make_inspector().evaluation())
print('GBT2:', gbt2.make_inspector().evaluation())
print('All TFDF models trained!')

In [ ]:
# Variable Importances (same as house-prices notebook)
print('Available variable importances:')
for importance in inspector_rf.variable_importances().keys():
    print('\t', importance)

variable_importance_metric = 'NUM_AS_ROOT'
variable_importances = inspector_rf.variable_importances()[variable_importance_metric]
feature_names_vi = [vi[0].name for vi in variable_importances]
feature_importances = [vi[1] for vi in variable_importances]
feature_ranks = range(len(feature_names_vi))

plt.figure(figsize=(12, 8))
bar = plt.barh(feature_ranks, feature_importances, label=[str(x) for x in feature_ranks])
plt.yticks(feature_ranks, feature_names_vi)
plt.gca().invert_yaxis()
for importance, patch in zip(feature_importances, bar.patches):
    plt.text(patch.get_x() + patch.get_width(), patch.get_y(), f'{importance:.4f}', va='top')
plt.xlabel(variable_importance_metric)
plt.title('Feature Importance: NUM_AS_ROOT')
plt.tight_layout()
plt.show()

In [ ]:
# TFDF Predictions + Ensemble
pred_rf = rf.predict(test_ds).squeeze()
pred_gbt = gbt.predict(test_ds).squeeze()
pred_rf2 = rf2.predict(test_ds).squeeze()
pred_gbt2 = gbt2.predict(test_ds).squeeze()

# OOF on training data for weight optimization
train_ds_pred = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_tourn[features].copy(), task=tfdf.keras.Task.REGRESSION)

oof_rf = rf.predict(train_ds_pred).squeeze()
oof_gbt = gbt.predict(train_ds_pred).squeeze()
oof_rf2 = rf2.predict(train_ds_pred).squeeze()
oof_gbt2 = gbt2.predict(train_ds_pred).squeeze()

y = train_tourn[LABEL].values.astype(np.float32)

print('Individual TFDF model train RMSE:')
for nm, p in [('RF', oof_rf), ('GBT', oof_gbt), ('RF2', oof_rf2), ('GBT2', oof_gbt2)]:
    r = np.sqrt(mean_squared_error(y, np.clip(p, 1, 68)))
    print(f'  {nm}: {r:.4f}')

In [ ]:
# XGBoost + CatBoost (extra models for mega-blend)
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
import optuna
from sklearn.preprocessing import RobustScaler
optuna.logging.set_verbosity(optuna.logging.WARNING)

X = train_tourn[features].values.astype(np.float32)
Xt = test_tourn[features].values.astype(np.float32)
seasons = train_tourn['Season'].values
scaler = RobustScaler()
gkf = GroupKFold(n_splits=5)

# Optuna-tuned XGBoost
def opt_xgb(trial):
    p = {'n_estimators': trial.suggest_int('ne', 300, 1000),
         'max_depth': trial.suggest_int('md', 3, 7),
         'learning_rate': trial.suggest_float('lr', 0.01, 0.1, log=True),
         'subsample': trial.suggest_float('ss', 0.5, 0.95),
         'colsample_bytree': trial.suggest_float('cs', 0.4, 0.9),
         'reg_alpha': trial.suggest_float('a', 0.01, 5, log=True),
         'reg_lambda': trial.suggest_float('l', 0.1, 5, log=True),
         'min_child_weight': trial.suggest_int('mcw', 1, 10),
         'device': 'cuda', 'tree_method': 'hist',
         'random_state': 42, 'n_jobs': -1}
    oof = np.zeros(len(X))
    for _, (tri, vai) in enumerate(gkf.split(X, y, seasons)):
        Xtr = scaler.fit_transform(X[tri]); Xva = scaler.transform(X[vai])
        m = xgb.XGBRegressor(**p); m.fit(Xtr, y[tri])
        oof[vai] = np.clip(m.predict(Xva), 1, 68)
    return np.sqrt(mean_squared_error(y, oof))

print('Tuning XGBoost (10 trials)...')
sx = optuna.create_study(direction='minimize')
sx.optimize(opt_xgb, n_trials=10)
print(f'  Best XGB OOF RMSE: {sx.best_value:.4f}')

# Train best XGB
bp = sx.best_params
xgb_cfg = {'n_estimators':bp['ne'],'max_depth':bp['md'],'learning_rate':bp['lr'],
           'subsample':bp['ss'],'colsample_bytree':bp['cs'],'reg_alpha':bp['a'],
           'reg_lambda':bp['l'],'min_child_weight':bp['mcw'],
           'device':'cuda','tree_method':'hist','random_state':42,'n_jobs':-1}

oof_xgb = np.zeros(len(X)); pred_xgb = np.zeros(len(Xt))
for fold, (tri, vai) in enumerate(gkf.split(X, y, seasons)):
    Xtr = scaler.fit_transform(X[tri]); Xva = scaler.transform(X[vai])
    m = xgb.XGBRegressor(**xgb_cfg); m.fit(Xtr, y[tri])
    oof_xgb[vai] = np.clip(m.predict(Xva), 1, 68)
    pred_xgb += np.clip(m.predict(scaler.transform(Xt)), 1, 68)/5
print(f'  XGB OOF RMSE: {np.sqrt(mean_squared_error(y, oof_xgb)):.4f}')

In [ ]:
# CatBoost with conference as categorical
conf_cats = list(train_tourn['Conference'].unique())
train_tourn['conf_idx'] = train_tourn['Conference'].apply(
    lambda c: conf_cats.index(c) if c in conf_cats else -1).astype(int)
test_tourn['conf_idx'] = test_tourn['Conference'].apply(
    lambda c: conf_cats.index(c) if c in conf_cats else -1).astype(int)
features_cb = features + ['conf_idx']
cat_idx = [features_cb.index('conf_idx')]

# Use DataFrames to preserve int dtype for cat features
X_cb_df = train_tourn[features_cb].copy()
Xt_cb_df = test_tourn[features_cb].copy()

print('Training CatBoost (5-fold, CPU fallback)...')
oof_cb = np.zeros(len(X)); pred_cb = np.zeros(len(Xt))
for fold, (tri, vai) in enumerate(gkf.split(X_cb_df.values, y, seasons)):
    train_pool = Pool(X_cb_df.iloc[tri], y[tri], cat_features=cat_idx)
    val_pool = Pool(X_cb_df.iloc[vai], cat_features=cat_idx)
    test_pool = Pool(Xt_cb_df, cat_features=cat_idx)
    m = CatBoostRegressor(iterations=300, depth=6, learning_rate=0.03,
                           l2_leaf_reg=3, task_type='CPU', random_seed=42, verbose=0)
    m.fit(train_pool)
    oof_cb[vai] = np.clip(m.predict(val_pool), 1, 68)
    pred_cb += np.clip(m.predict(test_pool), 1, 68)/5
print(f'  CatBoost OOF RMSE: {np.sqrt(mean_squared_error(y, oof_cb)):.4f}')

In [ ]:
# Mega-Blend: TFDF-only (4 models) for speed/stability
all_oof = np.column_stack([oof_rf, oof_gbt, oof_rf2, oof_gbt2])
all_test = np.column_stack([pred_rf, pred_gbt, pred_rf2, pred_gbt2])
model_names = ['RF','GBT','RF2','GBT2']
n_models = len(model_names)

def mega_loss(w):
    w = np.abs(w)/np.abs(w).sum()
    return np.sqrt(mean_squared_error(y, np.clip(all_oof @ w, 1, 68)))

best_mr, best_mw = 999, None
for s in range(200):
    x0 = np.random.RandomState(s).dirichlet(np.ones(n_models))
    r = minimize(mega_loss, x0=x0, method='Nelder-Mead', options={'maxiter':8000})
    if r.fun < best_mr:
        best_mr, best_mw = r.fun, np.abs(r.x)/np.abs(r.x).sum()

mega_test = np.clip(all_test @ best_mw, 1, 68)

print(f'Mega-Blend train RMSE: {best_mr:.4f}')
print('\nModel weights:')
for nm, w in zip(model_names, best_mw):
    if w > 0.01: print(f'  {nm}: {w:.1%}')

In [ ]:
# Hungarian Assignment + Submissions (final step)
non_tourn_ids = test_data[test_data['Bid Type'].isna()]['RecordID'].values

def make_submission(preds, fname, hungarian=True):
    s = sub_template.copy()
    pm = {r: 0 for r in non_tourn_ids}
    if hungarian:
        for season in sorted(test_data['Season'].unique()):
            ts = dataset_df[(dataset_df['Season']==season) & (dataset_df['Overall Seed'].notna())]
            known = set(ts['Overall Seed'].astype(int).values)
            available = sorted(set(range(1, 69)) - known)
            te = test_tourn[test_tourn['Season']==season]
            pos = [np.where(test_tourn.index==idx)[0][0] for idx in te.index]
            ps = preds[pos]
            nn = len(ps)
            cost = np.zeros((nn, len(available)))
            for i in range(nn):
                for j in range(len(available)):
                    cost[i, j] = (ps[i] - available[j]) ** 2
            ri, ci = linear_sum_assignment(cost)
            for i, j in zip(ri, ci):
                pm[te.iloc[i]['RecordID']] = available[j]
    else:
        for rid, p in zip(test_tourn['RecordID'].values, preds):
            pm[rid] = round(float(np.clip(p, 1, 68)), 4)
    s['Overall Seed'] = s['RecordID'].map(pm)
    s.to_csv(fname, index=False)
    nz = (s['Overall Seed'] > 0.5).sum()
    print(f'  {fname:45s}: mean={s["Overall Seed"].mean():.3f}, nonzero={nz}')
    return fname

print('Generating submissions...')
# TFDF-only ensemble
tfdf_w = np.array([0.25, 0.25, 0.25, 0.25])
tfdf_ens = np.clip(np.column_stack([pred_rf, pred_gbt, pred_rf2, pred_gbt2]) @ tfdf_w, 1, 68)

f1 = make_submission(mega_test, 'submission_mega_hungarian.csv', True)
f2 = make_submission(mega_test, 'submission_mega_continuous.csv', False)
f3 = make_submission(tfdf_ens, 'submission_tfdf_hungarian.csv', True)
print('\nSubmit ALL and pick the best:')
print('  1. submission_mega_hungarian.csv      (TFDF mega-blend + Hungarian) - likely best')
print('  2. submission_mega_continuous.csv      (TFDF mega-blend continuous)')
print('  3. submission_tfdf_hungarian.csv       (TFDF-only + Hungarian)')
for f in [f1, f2, f3]:
    files.download(f)

In [ ]:
# Inspect predictions (sanity check)
print('Sample predictions (top NET teams per season):')
hun_map = pd.read_csv('submission_mega_hungarian.csv')
hun_dict = dict(zip(hun_map['RecordID'], hun_map['Overall Seed']))

for season in sorted(test_data['Season'].unique()):
    te = test_tourn[test_tourn['Season']==season].copy()
    pos = [np.where(test_tourn.index==idx)[0][0] for idx in te.index]
    te['ML_pred'] = mega_test[pos]
    te['Hungarian'] = te['RecordID'].map(hun_dict)
    print(f'\n--- {season} ---')
    te_show = te.sort_values('NET')[['Team','NET','tourn_net_rank','ML_pred','Hungarian','Conference','Bid Type']].head(8)
    print(te_show.to_string(index=False))

In [ ]:
# Explicitly download the main submission; warn if missing
import os
from google.colab import files
candidates = [
    'submission_mega_hungarian.csv',
    'submission_mega_continuous.csv',
    'submission_tfdf_hungarian.csv'
]
for fname in candidates:
    if os.path.exists(fname):
        print(f'Downloading {fname} ...')
        files.download(fname)
    else:
        print(f'Missing {fname} — rerun the submission cell above if needed.')

In [ ]:
# Copy submissions to Google Drive (if they exist)
import os, shutil
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
dest = '/content/drive/MyDrive/NCAA_submissions'
os.makedirs(dest, exist_ok=True)
candidates = [
    'submission_mega_hungarian.csv',
    'submission_mega_continuous.csv',
    'submission_tfdf_hungarian.csv'
]
for fname in candidates:
    if os.path.exists(fname):
        shutil.copy(fname, dest)
        print(f'Copied {fname} -> {dest}')
    else:
        print(f'Missing {fname}; rerun the submission cell above if needed.')